# 提交说明（Chapter 4）

本 notebook 对应“近似匹配与增量回填”机制实现。

- 运行前请先配置参数单元。
- 仅使用脱敏样例数据进行公开提交。
- 提交前清理所有输出。

In [ ]:
# 参数区（提交版本）
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("CH5_PROJECT_ROOT", "."))
DATA_ROOT = Path(os.environ.get("CH5_DATA_ROOT", "./data_sample"))
OUTPUT_ROOT = Path(os.environ.get("CH5_OUTPUT_ROOT", "./outputs"))

for p in [PROJECT_ROOT, DATA_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
import pandas as pd
import difflib

In [ ]:
raw_path = str(DATA_ROOT.as_posix()) + '/'
df_raw = pd.read_excel(raw_path + 'CHPA.xlsx')

In [ ]:
df = df_raw.copy()  # Replace with df = df_raw.copy()

In [ ]:
#df = df_raw.loc[df_raw['Manu'].str.contains('赫力昂|汤臣倍健|强生|科赴|J&J|Kenvue', na=False)].copy()

### Manu Mapping

In [ ]:
import pandas as pd
from rapidfuzz import fuzz
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache

# 假设 df 已经读取完毕
# 第一步：针对 Channel 为 O2O 和 EC 的项目，将 Manu_Mapped, Brand_Mapped, SKU_Mapped 直接等于对应的 Manu, Brand, SKU
mask = df['Channel'].isin(['O2O', 'EC'])
df.loc[mask, 'Manu_Mapped'] = df.loc[mask, 'Manu']
df.loc[mask, 'Brand_Mapped'] = df.loc[mask, 'Brand']
df.loc[mask, 'SKU_Mapped'] = df.loc[mask, 'SKU']

# 准备基准数据：Channel 为 EC 和 O2O 的唯一 Manu 值
base_manu_list = df[df['Channel'].isin(['EC', 'O2O'])]['Manu'].dropna().astype(str).unique().tolist()

# 获取 Pharmacy 行的唯一 Manu 值
pharmacy_manu_list = df[df['Channel'] == 'Pharmacy']['Manu'].dropna().astype(str).unique().tolist()

# 定义相似度阈值
SIMILARITY_THRESHOLD = 0.9

# 缓存相似度计算
@lru_cache(maxsize=10000)
def compute_similarity(str1, str2):
    # 使用 token_sort_ratio 和 partial_ratio 的加权平均
    token_score = fuzz.token_sort_ratio(str1, str2)
    partial_score = fuzz.partial_ratio(str1, str2)
    return (0.6 * token_score + 0.4 * partial_score) / 100.0  # 转换为 0-1 范围

# 定义匹配函数：为单个 Pharmacy Manu 找到最佳模糊匹配的 Manu
def match_manu(pharmacy_manu):
    max_sim = 0
    best_manu = None
    
    # 遍历基准 Manu 列表
    for base_manu in base_manu_list:
        manu_sim = compute_similarity(pharmacy_manu, base_manu)
        if manu_sim > max_sim:
            max_sim = manu_sim
            best_manu = base_manu
    
    if max_sim >= SIMILARITY_THRESHOLD:
        return pharmacy_manu, best_manu
    return pharmacy_manu, pharmacy_manu  # 对于未匹配的，返回原始 Manu

# 使用多线程进行唯一 Manu 的匹配
manu_matches = {}
if pharmacy_manu_list:
    with ThreadPoolExecutor(max_workers=20) as executor:  # 可调整线程数，根据 CPU 核心
        # 提交任务
        futures = [executor.submit(match_manu, manu) for manu in pharmacy_manu_list]
        # 使用 tqdm 监控进度
        for future in tqdm(as_completed(futures), total=len(pharmacy_manu_list), desc="Matching Unique Pharmacy Manu"):
            pharmacy_manu, matched_manu = future.result()
            manu_matches[pharmacy_manu] = matched_manu

# 更新 df 中的 Manu_Mapped
mask_pharmacy = df['Channel'] == 'Pharmacy'
df.loc[mask_pharmacy, 'Manu_Mapped'] = df.loc[mask_pharmacy, 'Manu'].map(manu_matches)

### Brand Mapping

In [ ]:
df = df_raw.copy()  # Replace with df = df_raw.copy()

In [ ]:
#df = df_raw.loc[df_raw['Manu'].str.contains('赫力昂|汤臣倍健|强生|科赴|J&J|Kenvue', na=False)].copy()

In [ ]:
import pandas as pd
from rapidfuzz import fuzz
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
import re

# 假设 df 已经读取完毕，且 Manu_Mapped 已匹配完毕
# 第一步：针对 Channel 为 O2O 和 EC 的项目，确保 Brand_Mapped 已设置
mask = df['Channel'].isin(['O2O', 'EC'])
df.loc[mask, 'Brand_Mapped'] = df.loc[mask, 'Brand']

# 预处理函数：清理品牌名称，移除常见后缀和标点（可选，根据需要调整）
def preprocess_brand(brand):
    if pd.isna(brand):
        return ""
    brand = str(brand)
    brand = re.sub(r'[()（）]', '', brand)  # 移除括号
    # 可以添加更多品牌特定的清理规则，如果必要
    return brand.strip()

# 准备基准数据：Channel 为 EC 的唯一 (Manu, Brand) 组合
base_df = df[df['Channel'] == 'EC'][['Manu', 'Brand']].dropna().drop_duplicates()
base_df['Manu'] = base_df['Manu'].astype(str)
base_df['Brand'] = base_df['Brand'].astype(str)
# 预处理基准 Brand
base_df['Brand_preprocessed'] = base_df['Brand'].apply(preprocess_brand)

# 获取 Pharmacy 行的唯一 (Manu_Mapped, Brand) 组合
pharmacy_pairs = df[df['Channel'] == 'Pharmacy'][['Manu_Mapped', 'Brand']].dropna().drop_duplicates()
pharmacy_pairs['Manu_Mapped'] = pharmacy_pairs['Manu_Mapped'].astype(str)
pharmacy_pairs['Brand'] = pharmacy_pairs['Brand'].astype(str)
pharmacy_pairs_list = pharmacy_pairs.to_records(index=False)

# 定义相似度阈值
SIMILARITY_THRESHOLD = 0.9

# 缓存相似度计算
@lru_cache(maxsize=10000)
def compute_similarity(str1, str2):
    # 使用 token_set_ratio 和 partial_ratio 的加权平均
    token_score = fuzz.token_set_ratio(str1, str2)
    partial_score = fuzz.partial_ratio(str1, str2)
    return (0.7 * token_score + 0.3 * partial_score) / 100.0  # 转换为 0-1 范围

# 定义匹配函数：为单个 Pharmacy (Manu_Mapped, Brand) 找到最佳模糊匹配的 Brand
def match_brand_pair(manu_mapped, brand):
    preprocessed_brand = preprocess_brand(brand)
    # 过滤基准数据为 Manu == manu_mapped
    filtered_base = base_df[base_df['Manu'] == manu_mapped]
    if filtered_base.empty:
        return (manu_mapped, brand), brand  # 无匹配 Manu，返回原始 Brand
    
    max_sim = 0
    best_brand = None
    
    # 遍历过滤后的基准 Brand
    for _, row in filtered_base.iterrows():
        brand_sim = compute_similarity(preprocessed_brand, row['Brand_preprocessed'])
        if brand_sim > max_sim:
            max_sim = brand_sim
            best_brand = row['Brand']  # 返回原始 Brand
    
    if max_sim >= SIMILARITY_THRESHOLD:
        return (manu_mapped, brand), best_brand
    return (manu_mapped, brand), brand  # 未达到阈值，返回原始 Brand

# 使用多线程进行唯一组合的匹配
brand_matches = {}
if len(pharmacy_pairs_list) > 0:
    with ThreadPoolExecutor(max_workers=20) as executor:  # 可调整线程数，根据 CPU 核心
        # 提交任务
        futures = [executor.submit(match_brand_pair, manu_mapped, brand) for manu_mapped, brand in pharmacy_pairs_list]
        # 使用 tqdm 监控进度
        for future in tqdm(as_completed(futures), total=len(pharmacy_pairs_list), desc="Matching Unique Pharmacy Brands"):
            pair, matched_brand = future.result()
            brand_matches[pair] = matched_brand

# 更新 df 中的 Brand_Mapped
def map_brand(row):
    if row['Channel'] == 'Pharmacy':
        key = (str(row['Manu_Mapped']), str(row['Brand']))
        return brand_matches.get(key, row['Brand'])  # 如果不在唯一组合中，返回原始
    return row['Brand_Mapped']

df['Brand_Mapped'] = df.apply(map_brand, axis=1)

# 输出或保存 df（根据需要）
print(df)
# df.to_csv('output.csv', index=False)  # 如果需要保存

In [ ]:
df.loc[(df['Channel'].str.contains('Pharmacy|EC')) & (df['Brand'].str.contains('美林')), ['SKU','Brand', 'Brand_Mapped']]
#df.loc[(df['Channel'] == 'Pharmacy'), ['SKU','SKU_ID', 'SKU_Mapped','Manu_Mapped','Brand_Mapped']]

In [ ]:
### Brand Mapping

In [ ]:
### Export

In [ ]:
df.to_excel(raw_path + 'Omni_Channel_SKU_Mapped.xlsx', index=False)

### END